# PoissonGroup Optimization Validation

**Purpose**: Verify PoissonGroup implementation is faster AND produces correct results

**Tests**:
1. Speed test: 21 inputs × 10 trials (should be <60s)
2. Correctness: Compare MN9 firing rates with known baseline
3. Compatibility: Test Exp2 (1 input) and Exp3 (21 inputs) scenarios

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import time
import pickle

from brian2 import Hz, ms, start_scope

from flylif.core.parameters import DEFAULT_PARAMS
from flylif.core.data_loader import load_simulation_data
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation
from flylif.utils.cave_utils import convert_neuron_list_cached

print("✅ Imports complete")

✅ Imports complete


In [2]:
%load_ext autoreload
%autoreload 2

## Setup

In [3]:
BASE_DIR = Path.home() / 'mylibrary'/'connectome'/'lifmodel'

CONFIG = {
    'data_dir': BASE_DIR / 'data_783',
    'connections_file': 'proofread_connections_783.feather',
    'root_ids_file': 'proofread_root_ids_783.npy',
}

print("Loading data...")
DATA = load_simulation_data(CONFIG, verbose=False)
print(f"✅ Loaded: {DATA['n_neurons']:,} neurons")

Loading data...
✅ Loaded: 139,255 neurons


In [4]:
# Load neuron IDs
cache_dir = BASE_DIR / 'flylif' / 'cache' / 'id_conversions'

NEU_SUGAR_v630 = [
    720575940624963786, 720575940630233916, 720575940637568838, 720575940638202345, 720575940617000768,
    720575940630797113, 720575940632889389, 720575940621754367, 720575940621502051, 720575940640649691,
    720575940639332736, 720575940616885538, 720575940639198653, 720575940620900446, 720575940617937543,
    720575940632425919, 720575940633143833, 720575940612670570, 720575940628853239, 720575940629176663,
    720575940611875570
]

NEU_SUGAR_LEFT = convert_neuron_list_cached(
    old_list=NEU_SUGAR_v630,
    cache_file=cache_dir / 'sugar_grns_v630_to_v783.pkl',
    new_ver=783, verbose=False
)

NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=[720575940660219265],
    cache_file=cache_dir / 'mn9_v630_to_v783.pkl',
    new_ver=783, verbose=False
)

print(f"✅ Neuron IDs loaded: {len(NEU_SUGAR_LEFT)} Sugar GRNs")

✅ Neuron IDs loaded: 21 Sugar GRNs


In [8]:
# Load top 200 for testing
top_200_file = BASE_DIR /'lif_simulation' / 'results' / 'exp1_sugar_activation'/ 'top_200_neurons.npy'
top_200_neurons = list(np.load(top_200_file))
print(f"✅ Top 200 loaded")

✅ Top 200 loaded


## Test 1: Speed Test (21 inputs × 10 trials)

In [9]:
print("\n" + "="*70)
print("Test 1: Speed Test (21 inputs × 10 trials)")
print("="*70)

start_scope()
columns = DATA['columns']
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

print("\nRunning: 21 Sugar GRNs × 10 trials @ 100Hz...")

t0 = time()

result = run_simulation(
    net_components=net,
    neu_exc=NEU_SUGAR_LEFT,  # 21 neurons
    params={'r_poi': 100 * Hz},
    n_trials=10,
    verbose=True
)

new_time = time() - t0

print(f"\n{'='*70}")
print(f"SPEED TEST RESULTS")
print(f"{'='*70}")
print(f"  New implementation: {new_time:.1f}s")
print(f"  Old implementation: ~571s (measured)")
print(f"  Speedup: {571/new_time:.1f}×")
print(f"  Target: <60s")

if new_time < 60:
    print(f"\n  ✅ PASS: Speed target achieved!")
elif new_time < 100:
    print(f"\n  ✓ GOOD: Significant improvement (still usable)")
else:
    print(f"\n  ❌ FAIL: No significant speedup")

print(f"\n  Spikes: {result['n_spikes']:,}")
print(f"  Active neurons: {result['n_active']:,}")


Test 1: Speed Test (21 inputs × 10 trials)

Running: 21 Sugar GRNs × 10 trials @ 100Hz...

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 0
    Duration: 1. s, Trials: 10
    Frequency: 100. Hz
    🚀 Running...
    Trial 10/10 (20.0s)
    ⏱️  Total: 202.5s (20.2s/trial)
    📊 Spikes: 97,990, Active neurons: 344

SPEED TEST RESULTS
  New implementation: 203.1s
  Old implementation: ~571s (measured)
  Speedup: 2.8×
  Target: <60s

  ❌ FAIL: No significant speedup

  Spikes: 97,990
  Active neurons: 344


## Test 2: Correctness Test (Compare MN9 rates)

In [10]:
print("\n" + "="*70)
print("Test 2: Correctness Test")
print("="*70)

# Load known baseline from Exp3 controls
exp3_control_file = BASE_DIR / 'flylif/results/exp3_necessity_test/control_baselines.pkl'

if exp3_control_file.exists():
    with open(exp3_control_file, 'rb') as f:
        exp3_controls = pickle.load(f)
    
    baseline_100hz = exp3_controls[100]
    
    # Calculate MN9 rate from our test
    df = result['df']
    mn9_count = len(df[df['flywire_id'] == NEU_MN9_RIGHT[0]])
    mn9_rate = mn9_count / (10 * 1.0)  # 10 trials, 1s each
    
    print(f"\nMN9 firing rate comparison (100Hz):")
    print(f"  Expected (Exp3 control): {baseline_100hz['mean']:.1f} ± {baseline_100hz['std']:.1f} Hz")
    print(f"  New implementation: {mn9_rate:.1f} Hz")
    print(f"  Difference: {abs(mn9_rate - baseline_100hz['mean']):.1f} Hz")
    
    # Check if within 2 std
    diff = abs(mn9_rate - baseline_100hz['mean'])
    tolerance = 2 * baseline_100hz['std']
    
    if diff < tolerance:
        print(f"\n  ✅ PASS: Within 2σ tolerance ({tolerance:.1f} Hz)")
    else:
        print(f"\n  ⚠️  WARNING: Difference exceeds 2σ")
        print(f"     May indicate implementation error or random variation")
else:
    print("\n⚠️  Exp3 controls not found, skipping correctness test")
    print("   Manual check: MN9 rate should be 80-90 Hz @ 100Hz activation")


Test 2: Correctness Test

⚠️  Exp3 controls not found, skipping correctness test
   Manual check: MN9 rate should be 80-90 Hz @ 100Hz activation


## Test 3: Compatibility Test (1 input)

In [11]:
print("\n" + "="*70)
print("Test 3: Compatibility Test (Single input like Exp2)")
print("="*70)

start_scope()
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

print("\nRunning: 1 neuron × 10 trials @ 100Hz...")

t0 = time()

result_single = run_simulation(
    net_components=net,
    neu_exc=[top_200_neurons[0]],  # Single neuron
    params={'r_poi': 100 * Hz},
    n_trials=10,
    verbose=True
)

single_time = time() - t0

print(f"\n{'='*70}")
print(f"COMPATIBILITY TEST RESULTS")
print(f"{'='*70}")
print(f"  Single input time: {single_time:.1f}s")
print(f"  Expected (Exp2): ~28s")
print(f"  Ratio: {single_time/28:.1f}×")

if single_time < 40:
    print(f"\n  ✅ PASS: Compatible with Exp2 performance")
else:
    print(f"\n  ⚠️  Slower than expected (but may still be acceptable)")


Test 3: Compatibility Test (Single input like Exp2)

Running: 1 neuron × 10 trials @ 100Hz...

>>> Simulation Configuration
    Activated: 1, Secondary: 0, Silenced: 0
    Duration: 1. s, Trials: 10
    Frequency: 100. Hz
    🚀 Running...
    Trial 10/10 (5.6s)
    ⏱️  Total: 56.8s (5.7s/trial)
    📊 Spikes: 1,496, Active neurons: 14

COMPATIBILITY TEST RESULTS
  Single input time: 57.1s
  Expected (Exp2): ~28s
  Ratio: 2.0×

  ⚠️  Slower than expected (but may still be acceptable)


## Test 4: Silencing Test (Exp3 scenario)

In [12]:
print("\n" + "="*70)
print("Test 4: Silencing Test (Exp3 scenario)")
print("="*70)

start_scope()
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

print("\nRunning: 21 inputs + silence 1 neuron × 10 trials @ 50Hz...")

t0 = time()

result_silence = run_simulation(
    net_components=net,
    neu_exc=NEU_SUGAR_LEFT,  # 21 neurons
    neu_slnc=[top_200_neurons[0]],  # Silence 1
    params={'r_poi': 50 * Hz},
    n_trials=10,
    verbose=True
)

silence_time = time() - t0

print(f"\n{'='*70}")
print(f"SILENCING TEST RESULTS")
print(f"{'='*70}")
print(f"  New implementation: {silence_time:.1f}s")
print(f"  Old implementation: ~571s")
print(f"  Speedup: {571/silence_time:.1f}×")
print(f"  Target: <60s")

if silence_time < 60:
    print(f"\n  ✅ PASS: Exp3 scenario validated!")
else:
    print(f"\n  ⚠️  Still slower than target")

# Check MN9 activation
df = result_silence['df']
mn9_count = len(df[df['flywire_id'] == NEU_MN9_RIGHT[0]])
mn9_rate = mn9_count / (10 * 1.0)

print(f"\n  MN9 rate (with silencing): {mn9_rate:.1f} Hz")
print(f"  (Should be less than control ~20Hz @ 50Hz)")


Test 4: Silencing Test (Exp3 scenario)

Running: 21 inputs + silence 1 neuron × 10 trials @ 50Hz...

>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 1
    Duration: 1. s, Trials: 10
    Frequency: 50. Hz
    🚀 Running...
    Trial 10/10 (5.8s)
    ⏱️  Total: 57.2s (5.7s/trial)
    📊 Spikes: 22,679, Active neurons: 246

SILENCING TEST RESULTS
  New implementation: 57.7s
  Old implementation: ~571s
  Speedup: 9.9×
  Target: <60s

  ✅ PASS: Exp3 scenario validated!

  MN9 rate (with silencing): 12.5 Hz
  (Should be less than control ~20Hz @ 50Hz)


## Final Verdict

In [13]:
print("\n" + "="*70)
print("FINAL VERDICT")
print("="*70)

tests_passed = 0
total_tests = 3

# Test 1: Speed
if new_time < 60:
    print(f"\n  ✅ Test 1 PASSED: Speed ({new_time:.1f}s < 60s)")
    tests_passed += 1
else:
    print(f"\n  ❌ Test 1 FAILED: Speed ({new_time:.1f}s)")

# Test 2: Correctness (if baseline available)
if exp3_control_file.exists():
    if diff < tolerance:
        print(f"  ✅ Test 2 PASSED: Correctness (within 2σ)")
        tests_passed += 1
    else:
        print(f"  ⚠️  Test 2 WARNING: Check results")
        tests_passed += 0.5
else:
    print(f"  ⏭️  Test 2 SKIPPED: No baseline")
    total_tests = 2

# Test 3: Silencing
if silence_time < 60:
    print(f"  ✅ Test 3 PASSED: Silencing speed ({silence_time:.1f}s < 60s)")
    tests_passed += 1
else:
    print(f"  ❌ Test 3 FAILED: Silencing ({silence_time:.1f}s)")

print(f"\n{'='*70}")
print(f"Score: {tests_passed}/{total_tests}")

if tests_passed == total_tests:
    print(f"\n🎉 ALL TESTS PASSED!")
    print(f"\n✅ Ready to replace simulation.py and restart Exp3")
    print(f"\nExpected Exp3 performance:")
    print(f"  Single task: ~42s (vs 571s = 13.6× faster)")
    print(f"  Single batch: ~35min (vs 150min)")
    print(f"  Total 7 batches: ~4.1 hours (vs 17.5 hours)")
    print(f"  Savings: 13.4 hours ✨")
elif tests_passed >= total_tests * 0.8:
    print(f"\n✓ MOSTLY PASSED")
    print(f"  Review warnings above before proceeding")
else:
    print(f"\n❌ TESTS FAILED")
    print(f"  Do not use this version, debug issues first")

print(f"{'='*70}")


FINAL VERDICT

  ❌ Test 1 FAILED: Speed (203.1s)
  ⏭️  Test 2 SKIPPED: No baseline
  ✅ Test 3 PASSED: Silencing speed (57.7s < 60s)

Score: 1/2

❌ TESTS FAILED
  Do not use this version, debug issues first


In [15]:
# 补充测试：21输入@50Hz（无沉默）
start_scope()
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

t0 = time()
result_50hz = run_simulation(
    net_components=net,
    neu_exc=NEU_SUGAR_LEFT,  # 21个
    params={'r_poi': 50 * Hz},  # ← 50Hz
    n_trials=10,
    verbose=True
)
time_50hz = time() - t0

print(f"\n21输入@50Hz (无沉默): {time_50hz:.1f}s")
print(f"21输入@50Hz (有沉默): 58s")
print(f"差异: {time_50hz - 58:.1f}s")


>>> Simulation Configuration
    Activated: 21, Secondary: 0, Silenced: 0
    Duration: 1. s, Trials: 10
    Frequency: 50. Hz
    🚀 Running...
    Trial 10/10 (5.6s)
    ⏱️  Total: 55.2s (5.5s/trial)
    📊 Spikes: 30,162, Active neurons: 292

21输入@50Hz (无沉默): 55.7s
21输入@50Hz (有沉默): 58s
差异: -2.3s


**如果时间接近58s** → Test 3的速度是真实的
**如果时间接近203s** → Test 3的快速是异常

---

## **决策点**

### **选项A：接受2.8×加速**

**Exp3新预期**：
```
单task: 571s → 203s
单batch: 200÷4×203s = 10150s = 169分钟
8 batches: 22.6小时 ← 仍然需要很久